In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')


In [12]:
df=pd.read_csv('sample_data/qoute_dataset.csv')


In [13]:
df.head()

,quote,Author
0,“The world as we have created it is a process ...,Albert Einstein
1,"“It is our choices, Harry, that show what we t...",J.K. Rowling
2,“There are only two ways to live your life. On...,Albert Einstein
3,"“The person, be it gentleman or lady, who has ...",Jane Austen
4,"“Imperfection is beauty, madness is genius and...",Marilyn Monroe


In [14]:
df.shape

(3038, 2)

In [15]:
quotes=df['quote']

In [16]:
quotes.head()

,quote
0,“The world as we have created it is a process ...
1,"“It is our choices, Harry, that show what we t..."
2,“There are only two ways to live your life. On...
3,"“The person, be it gentleman or lady, who has ..."
4,"“Imperfection is beauty, madness is genius and..."


In [17]:
quotes=quotes.str.lower()

In [18]:
import string
translator=str.maketrans('','',string.punctuation)
quotes=quotes.apply(lambda x:x.translate(translator))

In [19]:
quotes.head()

,quote
0,“the world as we have created it is a process ...
1,“it is our choices harry that show what we tru...
2,“there are only two ways to live your life one...
3,“the person be it gentleman or lady who has no...
4,“imperfection is beauty madness is genius and ...


In [20]:
from tensorflow.keras.preprocessing.text import Tokenizer


In [21]:
vocab_size=10000
token=Tokenizer(num_words=vocab_size)
token.fit_on_texts(quotes)

In [22]:
word_index=token.word_index
print(len(word_index))
list(word_index.items())[:10]

8978


[('the', 1),
 ('you', 2),
 ('to', 3),
 ('and', 4),
 ('a', 5),
 ('i', 6),
 ('is', 7),
 ('of', 8),
 ('that', 9),
 ('it', 10)]

In [23]:
sequence=token.texts_to_sequences(quotes)

In [24]:
quotes[0]

'“the world as we have created it is a process of our thinking it cannot be changed without changing our thinking”'

In [25]:
sequence[0]

[713,
 62,
 29,
 19,
 16,
 946,
 10,
 7,
 5,
 1156,
 8,
 70,
 293,
 10,
 145,
 12,
 809,
 104,
 752,
 70,
 2461]

In [26]:

X=[]
y=[]

for seq in sequence:
  for i in range (1,len(seq)):
    X.append(seq[:i])
    y.append(seq[i])


In [27]:
len(X)

85271

In [28]:
len(y)


85271

In [29]:
max_len=max(len(x) for x in X)
print(max_len)

745


In [30]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [31]:
X_padded=pad_sequences(X,maxlen=max_len,padding='pre')

In [32]:
X_padded[0]

array([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
         0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   

In [33]:
y=np.array(y)

In [34]:
X_padded.shape

(85271, 745)

In [35]:
y.shape



(85271,)

In [36]:
from tensorflow.keras.utils import to_categorical
y_categorical = to_categorical(y, num_classes=vocab_size)
display(y_categorical.shape)

(85271, 10000)

In [37]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,SimpleRNN

In [38]:
embedding_dim=50
rnn_units=128

In [39]:
rnn_model=Sequential()
rnn_model.add(Embedding(vocab_size,embedding_dim,input_length=max_len-1))
rnn_model.add(SimpleRNN(rnn_units))
rnn_model.add(Dense(vocab_size,activation='softmax'))


In [37]:
rnn_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


In [38]:
rnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [39]:
lstm_model=Sequential()
lstm_model.add(Embedding(vocab_size,embedding_dim,input_length=max_len))
lstm_model.add(LSTM(units=rnn_units))
lstm_model.add(Dense(units=vocab_size,activation='softmax'))

In [40]:
lstm_model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [41]:
epochs=10
batch_size=128

In [42]:
history_rnn=rnn_model.fit(X_padded,y_categorical,epochs=epochs,batch_size=batch_size,validation_split=0.1)

Epoch 1/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 47s 71ms/step - accuracy: 0.0452 - loss: 6.7147 - val_accuracy: 0.0591 - val_loss: 6.5463
Epoch 2/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.0779 - loss: 6.1224 - val_accuracy: 0.0870 - val_loss: 6.3297
Epoch 3/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.1020 - loss: 5.7756 - val_accuracy: 0.0996 - val_loss: 6.2974
Epoch 4/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.1153 - loss: 5.5016 - val_accuracy: 0.1018 - val_loss: 6.3163
Epoch 5/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.1298 - loss: 5.2522 - val_accuracy: 0.1080 - val_loss: 6.3373
Epoch 6/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 65ms/step - accuracy: 0.1423 - loss: 5.0193 - val_accuracy: 0.1094 - val_loss: 6.4090
Epoch 7/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 39s 66ms/step - accuracy: 0.1552 - loss: 4.7994 - val_accuracy: 0.1112 - val_loss: 6.4826
Epoch 8/10
600/600 ━━━━━━━━━━━━━━━━━━━━ 40s 66ms/step - accuracy: 0.1712 - loss: 4.5926 - 

In [43]:
rnn_model.save('rnn_model.h5')

In [45]:
epochs=100
batch_size=128

history_lstm=lstm_model.fit(X_padded,y_categorical,epochs=epochs,batch_size=batch_size,validation_split=0.1)

Epoch 1/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 43s 59ms/step - accuracy: 0.0387 - loss: 6.7637 - val_accuracy: 0.0435 - val_loss: 6.6803
Epoch 2/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.0586 - loss: 6.3234 - val_accuracy: 0.0687 - val_loss: 6.5752
Epoch 3/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.0831 - loss: 6.0519 - val_accuracy: 0.0874 - val_loss: 6.4760
Epoch 4/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1001 - loss: 5.8259 - val_accuracy: 0.0964 - val_loss: 6.4159
Epoch 5/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1117 - loss: 5.6370 - val_accuracy: 0.1027 - val_loss: 6.4014
Epoch 6/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1217 - loss: 5.4634 - val_accuracy: 0.1062 - val_loss: 6.4141
Epoch 7/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 53ms/step - accuracy: 0.1295 - loss: 5.3088 - val_accuracy: 0.1072 - val_loss: 6.4359
Epoch 8/100
600/600 ━━━━━━━━━━━━━━━━━━━━ 32s 54ms/step - accuracy: 0.1371 - loss: 5

In [46]:
lstm_model.save("lstm_model.h5")

In [49]:
index_to_word={}

for word, index in word_index.items():
  index_to_word[index] = word


In [50]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [51]:
def predictor(model,tokenizer,text,max_len):
  text = text.lower()

  seq = tokenizer.texts_to_sequences([text])[0]
  seq = pad_sequences([seq], maxlen=max_len, padding='pre')

  pred = model.predict(seq,verbose = 0)
  pred_index = np.argmax(pred)
  return index_to_word[pred_index]


In [68]:
seed_text = "i want to have"
next_word = predictor(lstm_model,token,seed_text,max_len)
print(next_word)

the


In [69]:

def generate_text(model,tokenizer,seed_text,max_len,n_words):
  for _ in range(n_words):
    next_word = predictor(model,tokenizer,seed_text,max_len)
    if next_word == "":
      break
    seed_text += " " + next_word
  return seed_text


In [70]:
seed = "did you  "
generate_text = generate_text(lstm_model,token,seed,max_len,10)
print(generate_text)


did you   remember the book thief many things that you can never


In [72]:
import pickle

In [73]:
with open("tokenizer.pickle","wb") as f:
  pickle.dump(token,f)

In [74]:
with open("max_len.pickle","wb") as f:
  pickle.dump(max_len,f)

In [1]:
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
